# 1. Amazon Veri Hazırlama ve Temizleme
Bu notebook, Amazon Fine Food Reviews ham CSV verisinden filtrelenmiş ve dengelenmiş bir CSV oluşturur.

In [1]:
import os
import pandas as pd
import numpy as np
from sklearn.utils import resample
import config

print('=== Klasörler Oluşturuluyor ===')
for d in ['data', 'models', 'features', 'results']:
    os.makedirs(d, exist_ok=True)

=== Klasörler Oluşturuluyor ===


In [2]:
print('\n=== ADIM 1: Veriyi Okuma ===')
df = pd.read_csv(config.DATA_FILE)
toplam_ham = len(df)
print(f'Toplam satır sayısı: {len(df):,}')
print(f'Kolonlar: {list(df.columns)}')
df.head(3)


=== ADIM 1: Veriyi Okuma ===


Toplam satır sayısı: 568,454
Kolonlar: ['Id', 'ProductId', 'UserId', 'ProfileName', 'HelpfulnessNumerator', 'HelpfulnessDenominator', 'Score', 'Time', 'Summary', 'Text']


,Id,ProductId,UserId,ProfileName,HelpfulnessNumerator,HelpfulnessDenominator,Score,Time,Summary,Text
0,1,B001E4KFG0,A3SGXH7AUHU8GW,delmartian,1,1,5,1303862400,Good Quality Dog Food,I have bought several of the Vitality canned d...
1,2,B00813GRG4,A1D87F6ZCVE5NK,dll pa,0,0,1,1346976000,Not as Advertised,Product arrived labeled as Jumbo Salted Peanut...
2,3,B000LQOCH0,ABXLMWJIXXAIN,"Natalia Corres ""Natalia Corres""",1,1,4,1219017600,"""Delight"" says it all",This is a confection that has been around a fe...


In [3]:
print('\n=== ADIM 2: Veri Temizliği ===')
df = df.dropna(subset=['Text'])
df = df[df['Text'].astype(str).str.strip() != '']
df = df[df['Text'].str.len() >= 10]

initial = len(df)
df = df.drop_duplicates(subset=['UserId', 'Text'])
print(f'Duplikat kaldırma: {initial:,} -> {len(df):,} ({initial - len(df):,} silindi)')
print(f'Temizlik sonrası kalan: {len(df):,}')


=== ADIM 2: Veri Temizliği ===


Duplikat kaldırma: 568,454 -> 393,606 (174,848 silindi)
Temizlik sonrası kalan: 393,606


In [4]:
print('\n=== ADIM 3: Yıldız -> 3-Sınıf Dönüşümü ===')
print('Orijinal Score dağılımı:')
print(df['Score'].value_counts().sort_index())

def map_score(s):
    if s in [1, 2]: return 0
    elif s == 3:    return 1
    elif s in [4, 5]: return 2
    return -1

df['label'] = df['Score'].apply(map_score)
df = df[df['label'] != -1]

df = df.rename(columns={'Text': 'text', 'Summary': 'summary'})
df = df[['Id','ProductId','UserId','HelpfulnessNumerator',
         'HelpfulnessDenominator','Score','Time','summary','text','label']]
print('Dönüşüm tamamlandı.')


=== ADIM 3: Yıldız -> 3-Sınıf Dönüşümü ===
Orijinal Score dağılımı:
Score
1     36281
2     20795
3     29756
4     56044
5    250730
Name: count, dtype: int64


Dönüşüm tamamlandı.


In [5]:
print('\n=== ADIM 4: Undersampling ===')
print('Dengeleme Öncesi:')
cc = df['label'].value_counts()
for lbl, cnt in cc.items():
    print(f'  {config.CLASS_NAMES[lbl]} (Sınıf {lbl}): {cnt:,} (%{cnt/len(df)*100:.2f})')

min_c = cc.min()
print(f'\nEn az sınıf ({min_c:,}) baz alınarak undersampling...')

parts = []
for lbl in cc.index:
    parts.append(resample(df[df['label']==lbl], replace=False,
                          n_samples=min_c, random_state=config.RANDOM_STATE))
df_balanced = pd.concat(parts).sample(frac=1, random_state=config.RANDOM_STATE).reset_index(drop=True)

print('\nDengeleme Sonrası:')
for lbl, cnt in df_balanced['label'].value_counts().items():
    print(f'  {config.CLASS_NAMES[lbl]} (Sınıf {lbl}): {cnt:,} (%{cnt/len(df_balanced)*100:.2f})')


=== ADIM 4: Undersampling ===
Dengeleme Öncesi:
  Pozitif (Sınıf 2): 306,774 (%77.94)
  Negatif (Sınıf 0): 57,076 (%14.50)
  Nötr (Sınıf 1): 29,756 (%7.56)

En az sınıf (29,756) baz alınarak undersampling...



Dengeleme Sonrası:
  Negatif (Sınıf 0): 29,756 (%33.33)
  Nötr (Sınıf 1): 29,756 (%33.33)
  Pozitif (Sınıf 2): 29,756 (%33.33)


In [6]:
print('\n=== ADIM 5: Kaydetme ===')
df_balanced.to_csv('data/reviews_cleaned.csv', index=False)
print('Kaydedildi: data/reviews_cleaned.csv')


=== ADIM 5: Kaydetme ===


Kaydedildi: data/reviews_cleaned.csv


In [7]:
print('\n' + '='*50)
print('VERİ SETİ İSTATİSTİK RAPORU'.center(50))
print('='*50)
print(f'Ham veri satır sayısı           : {toplam_ham:,}')
print(f'Dengelenmiş veri satır sayısı   : {len(df_balanced):,}')
print(f'Benzersiz Ürün Sayısı           : {df_balanced["ProductId"].nunique():,}')
print(f'Benzersiz Kullanıcı Sayısı      : {df_balanced["UserId"].nunique():,}')
print(f'Ortalama Yorum Uzunluğu         : {df_balanced["text"].str.len().mean():.0f} karakter')
print('='*50)


           VERİ SETİ İSTATİSTİK RAPORU            
Ham veri satır sayısı           : 568,454
Dengelenmiş veri satır sayısı   : 89,268
Benzersiz Ürün Sayısı           : 31,739
Benzersiz Kullanıcı Sayısı      : 70,590
Ortalama Yorum Uzunluğu         : 471 karakter
